In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv("Heart_disease_cleveland_new.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,0,145,233,1,2,150,0,2.3,2,0,2,0
1,67,1,3,160,286,0,2,108,1,1.5,1,3,1,1
2,67,1,3,120,229,0,2,129,1,2.6,1,2,3,1
3,37,1,2,130,250,0,0,187,0,3.5,2,0,1,0
4,41,0,1,130,204,0,2,172,0,1.4,0,0,1,0


In [5]:
df = pd.get_dummies(df, columns=["cp","restecg","slope","thal"], drop_first=True)
df.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,target,cp_1,cp_2,cp_3,restecg_1,restecg_2,slope_1,slope_2,thal_2,thal_3
0,63,1,145,233,1,150,0,2.3,0,0,False,False,False,False,True,False,True,True,False
1,67,1,160,286,0,108,1,1.5,3,1,False,False,True,False,True,True,False,False,False
2,67,1,120,229,0,129,1,2.6,2,1,False,False,True,False,True,True,False,False,True
3,37,1,130,250,0,187,0,3.5,0,0,False,True,False,False,False,False,True,False,False
4,41,0,130,204,0,172,0,1.4,0,0,True,False,False,False,True,False,False,False,False


In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
num_cols = ["age","trestbps","chol","thalach","oldpeak"]
df[num_cols] = scaler.fit_transform(df[num_cols])

In [7]:
df.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,target,cp_1,cp_2,cp_3,restecg_1,restecg_2,slope_1,slope_2,thal_2,thal_3
0,0.948726,1,0.757525,-0.264900,1,0.017197,0,1.087338,0,0,False,False,False,False,True,False,True,True,False
1,1.392002,1,1.611220,0.760415,0,-1.821905,1,0.397182,3,1,False,False,True,False,True,True,False,False,False
2,1.392002,1,-0.665300,-0.342283,0,-0.902354,1,1.346147,2,1,False,False,True,False,True,True,False,False,True
3,-1.932564,1,-0.096170,0.063974,0,1.637359,0,2.122573,0,0,False,True,False,False,False,False,True,False,False
4,-1.489288,0,-0.096170,-0.825922,0,0.980537,0,0.310912,0,0,True,False,False,False,True,False,False,False,False


In [8]:
from sklearn.model_selection import train_test_split

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [10]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [11]:
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(X_train, y_train)


,priors,None
,var_smoothing,1e-09


In [12]:
def evaluate_model_with_thresholds(model, model_name, X_train, y_train, X_test, y_test):
    results = []

    # -------- TRAIN --------
    start_train = time.time()
    model.fit(X_train, y_train)
    end_train = time.time()
    training_time = end_train - start_train

    # -------- MODEL SIZE --------
    model_pickle = pickle.dumps(model)
    model_size_kb = len(model_pickle) / 1024

    # -------- PROBABILITIES --------
    test_probs = model.predict_proba(X_test)[:, 1]
    train_probs = model.predict_proba(X_train)[:, 1]

    # -------- THRESHOLD LOOP --------
    for threshold in np.arange(0.50, 0.951, 0.05):

        start_pred = time.time()
        pred_test = (test_probs >= threshold).astype(int)
        end_pred = time.time()
        prediction_time = end_pred - start_pred

        pred_train = (train_probs >= threshold).astype(int)

        # Metrics
        test_accuracy = accuracy_score(y_test, pred_test)
        train_accuracy = accuracy_score(y_train, pred_train)

        overfitting_gap = train_accuracy - test_accuracy
        balanced_acc = balanced_accuracy_score(y_test, pred_test)
        f1 = f1_score(y_test, pred_test)
        precision = precision_score(y_test, pred_test, zero_division=0)
        recall = recall_score(y_test, pred_test, zero_division=0)
        roc_auc = roc_auc_score(y_test, test_probs)

        results.append([
            model_name, threshold, model_size_kb, prediction_time, training_time,
            overfitting_gap, test_accuracy, balanced_acc, f1, precision, roc_auc, recall
        ])

    columns = [
        "Model", "Threshold", "Model Size (KB)", "Prediction Time (s)", "Training Time (s)",
        "Overfitting Gap", "Accuracy", "Balanced Accuracy", "F1 Score",
        "Precision", "ROC-AUC", "Recall"
    ]

    return pd.DataFrame(results, columns=columns)


In [13]:
import time
import pickle

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [14]:
from lad.lad import LADClassifier
lad = LADClassifier()


results_df = evaluate_model_with_thresholds(lad, "LAD", X_train, y_train, X_test, y_test)


In [15]:
results_df

,Model,Threshold,Model Size (KB),Prediction Time (s),Training Time (s),Overfitting Gap,Accuracy,Balanced Accuracy,F1 Score,Precision,ROC-AUC,Recall
0,LAD,0.50,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
1,LAD,0.55,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
2,LAD,0.60,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
3,LAD,0.65,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
4,LAD,0.70,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
5,LAD,0.75,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
6,LAD,0.80,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
7,LAD,0.85,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
8,LAD,0.90,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
9,LAD,0.95,23.168945,0.0,1.70619,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25


In [16]:
results_df[['Overfitting Gap', 'Accuracy', 'Balanced Accuracy', 'F1 Score', 'Precision', 'ROC-AUC', 'Recall']]

,Overfitting Gap,Accuracy,Balanced Accuracy,F1 Score,Precision,ROC-AUC,Recall
0,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
1,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
2,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
3,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
4,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
5,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
6,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
7,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
8,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25
9,0.099648,0.557377,0.573276,0.372093,0.727273,0.573276,0.25


In [21]:
def extract_lad_rules(lad_model, feature_names):
    rules = []

    # Positive class rules
    for pattern in lad_model.positive_patterns_:
        conditions = []
        for (feat_idx, lower, upper) in pattern:
            feat_name = feature_names[feat_idx]
            if lower == upper:
                conditions.append(f"{feat_name} = {lower}")
            else:
                conditions.append(f"{lower} ≤ {feat_name} ≤ {upper}")
        rule_str = " AND ".join(conditions)
        rules.append(("POSITIVE (Disease Present)", rule_str))

    # Negative class rules
    for pattern in lad_model.negative_patterns_:
        conditions = []
        for (feat_idx, lower, upper) in pattern:
            feat_name = feature_names[feat_idx]
            if lower == upper:
                conditions.append(f"{feat_name} = {lower}")
            else:
                conditions.append(f"{lower} ≤ {feat_name} ≤ {upper}")
        rule_str = " AND ".join(conditions)
        rules.append(("NEGATIVE (No Disease)", rule_str))

    return rules


In [27]:
import re

def map_att_to_names(rules_text, feature_names):
    def repl(match):
        idx = int(match.group(1))
        return feature_names[idx]  # replace attN with the feature name
    return re.sub(r"att(\d+)", repl, rules_text)

lad_rules_text = str(lad)
mapped_rules = map_att_to_names(lad_rules_text, feature_names)
print(mapped_rules)


MaxPatterns Set of Rules [151]:
0 → trestbps > -0.1815 AND chol > -0.2359 AND thalach > 0.1267 AND ca > 0.5 AND cp_3 > 0.5 AND restecg_2 > 0.5 AND thal_3 > 0.5
0 → age > 0.5609 AND oldpeak > -0.2498 AND ca > 0.5 AND cp_3 > 0.5 AND restecg_2 > 0.5 AND thal_3 > 0.5
0 → chol > -0.2359 AND thalach > 0.1267 AND oldpeak > -0.2498 AND ca > 0.5 AND cp_3 > 0.5
0 → chol > -0.2359 AND thalach > 0.1267 AND ca > 0.5 AND restecg_2 > 0.5 AND thal_3 > 0.5
0 → chol > -0.2359 AND thalach > 0.1267 AND ca > 0.5 AND cp_3 > 0.5 AND restecg_2 > 0.5 AND thal_3 > 0.5
0 → age > 0.5609 AND chol > -0.2359 AND oldpeak > -0.2498 AND cp_3 > 0.5 AND thal_3 > 0.5
0 → trestbps > -0.1815 AND chol > -0.2359 AND ca > 0.5 AND cp_3 > 0.5 AND restecg_2 > 0.5
0 → age > 0.5609 AND trestbps > -0.1815 AND chol > -0.2359 AND thalach > 0.1267 AND oldpeak > -0.2498 AND restecg_2 > 0.5
0 → trestbps > -0.1815 AND chol > -0.2359 AND ca > 0.5 AND cp_3 > 0.5 AND thal_3 > 0.5
0 → sex > 0.5 AND trestbps > -0.1815 AND chol > -0.2359 AND ol